# Данные и первый эксперимент

Авторы BelOAI, CC BY 4.0. Python 3, стандартная библиотека.

Скачайте `gliders.csv` из главы и загрузите рядом с ноутбуком. 24 синтетических планера: это учебная таблица, не физическая модель. Метка задаётся размахом; масса варьируется независимо.

Цель — получить baseline на validation. Test в этом ноутбуке не оцениваем.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Объекты, признаки и ответы](../../ml/first-experiment/objects-features/)
- [Разбиение данных и baseline](../../ml/first-experiment/split-baseline/)


In [ ]:
import csv
from pathlib import Path

# В Colab/Jupyter загрузите gliders.csv рядом с ноутбуком.
data_path = Path("gliders.csv")
if not data_path.is_file():
    raise FileNotFoundError("Загрузите gliders.csv в рабочую папку ноутбука")
with data_path.open(encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))
for row in rows:
    row["span_cm"] = float(row["span_cm"])
    row["mass_g"] = float(row["mass_g"])
train = [row for row in rows if row["split"] == "train"]
validation = [row for row in rows if row["split"] == "validation"]
test = [row for row in rows if row["split"] == "test"]
assert (len(train), len(validation), len(test)) == (12, 6, 6)
assert len({row["id"] for row in rows}) == len(rows)
print("train / validation / test:", len(train), len(validation), len(test))


In [ ]:
def accuracy(expected, predicted):
    if len(expected) == 0 or len(expected) != len(predicted):
        raise ValueError("Нужны непустые списки одинаковой длины")
    correct = 0
    for index in range(len(expected)):
        if expected[index] == predicted[index]:
            correct += 1
    return correct / len(expected)


Частоты считаем словарём только по train. Обходим названия классов в алфавитном порядке и заменяем победителя только при строго большей частоте. Поэтому при равенстве остаётся первая метка по алфавиту.

In [ ]:
counts = {}
for row in train:
    label = row["flight"]
    counts[label] = counts.get(label, 0) + 1
baseline = None
for label in sorted(counts):
    if baseline is None or counts[label] > counts[baseline]:
        baseline = label
expected = [row["flight"] for row in validation]
baseline_validation = accuracy(expected, [baseline] * len(validation))
assert baseline_validation == 0.5
print("Baseline:", baseline, "validation accuracy:", baseline_validation)


## Задачи

1. Посчитайте accuracy для `[0, 1, 1, 0]` и `[0, 0, 1, 0]`.
2. Проверьте непересечение идентификаторов частей.
3. Почему `flight` нельзя добавить в признаки? Сначала решите самостоятельно.

## Решения

Верны три из четырёх ответов. Идентификаторы частей должны быть различны; дополнительно исследуйте дубликаты реальных объектов. Использование `flight` как признака раскрывает ответ.

In [ ]:
assert accuracy([0, 1, 1, 0], [0, 0, 1, 0]) == 0.75
train_ids = {row["id"] for row in train}
validation_ids = {row["id"] for row in validation}
test_ids = {row["id"] for row in test}
assert not (train_ids & validation_ids)
assert not (train_ids & test_ids)
assert not (validation_ids & test_ids)
for part in (train, validation, test):
    part_counts = {}
    for row in part:
        label = row["flight"]
        part_counts[label] = part_counts.get(label, 0) + 1
    print("Классы:", part_counts)

## Вывод

Baseline на validation — 0.5. Не выбирайте решения по test. Перейдите к векторам и расстояниям, затем к KNN.